# 1. Modelo de FitzHugh-Nagumo / Hodgkin-Huxley (Pulsos Eléctricos Neuronales)

Simula el disparo eléctrico que permite a las neuronas transmitir información o al corazón latir a un ritmo coordinado.Al mover el control I (Corriente eléctrica de estímulo):Si fijas I cerca de 0, la línea de voltaje (\(V\)) se queda plana abajo. El estímulo es demasiado débil y la neurona permanece dormida (estado de reposo).Si subes I por encima de 0.4 o 0.5, el sistema rompe el equilibrio y el voltaje (\(V\), línea morada) empieza a oscilar formando ondas repetitivas periódicas. Esto representa trenes de potenciales de acción (disparos neuronales continuos).Al mover el control tau (Retraso de Canales): Controla el tiempo que tardan las compuertas iónicas de potasio (\(W\), línea verde punteada) en abrirse para contrarrestar el voltaje. Si aumentas tau, el canal tarda más en reaccionar, lo que ensancha los pulsos eléctricos y reduce la frecuencia del latido o disparo.

**Nota técnica:**

 El modelo original de Hodgkin-Huxley requiere 4 ecuaciones diferenciales complejas muy sensibles al paso del tiempo. Para garantizar estabilidad interactiva fluida en VS Code, se utiliza el modelo de FitzHugh-Nagumo, la simplificación matemática estándar más famosa que retiene la esencia del disparo de potencial de acción neuronal midiendo el voltaje (\(V\)) y la recuperación de canales iónicos (\(W\)).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from ipywidgets import interact, FloatSlider

# Ecuaciones del modelo FitzHugh-Nagumo (Hodgkin-Huxley simplificado)
def fitzhugh_nagumo(y, t, I, a, b, tau):
    V, W = y
    dVdt = V - (V**3)/3 - W + I
    dWdt = (V + a - b * W) / tau
    return [dVdt, dWdt]

def plot_fhn(I, tau):
    t = np.linspace(0, 100, 1000)
    y0 = [-1.0, -0.5] # Estado de reposo inicial
    
    # Parámetros estructurales estables
    a = 0.7
    b = 0.8
    
    sol = odeint(fitzhugh_nagumo, y0, t, args=(I, a, b, tau))
    
    plt.figure(figsize=(10, 5))
    plt.plot(t, sol[:, 0], lw=2.5, color='purple', label='Voltaje de Membrana (V)')
    plt.plot(t, sol[:, 1], lw=2, color='seagreen', label='Recuperación de Canales (W)', linestyle='--')
    
    plt.title('Potencial de Acción Neuronal (FitzHugh-Nagumo / Hodgkin-Huxley)', fontsize=14, fontweight='bold')
    plt.xlabel('Tiempo', fontsize=12)
    plt.ylabel('Estado de la Membrana', fontsize=12)
    plt.axhline(y=0, color='gray', linewidth=0.5, linestyle=':')
    plt.grid(True, alpha=0.3)
    plt.legend(loc='upper right')
    plt.ylim(-2.5, 2.5)
    plt.show()

# Controles interactivos
# Modifica 'I' (Corriente eléctrica) a más de 0.4 para ver a la neurona disparar pulsos repetidamente
interact(plot_fhn, 
         I=FloatSlider(value=0.5, min=0.0, max=2.0, step=0.05, description='Corriente (I):'),
         tau=FloatSlider(value=12.5, min=5.0, max=25.0, step=0.5, description='Retraso Canales:'))


interactive(children=(FloatSlider(value=0.5, description='Corriente (I):', max=2.0, step=0.05), FloatSlider(va…

<function __main__.plot_fhn(I, tau)>


# 2. Extensión de Hodgkin-Huxley (Concepto de Retroalimentación): 

**El Interruptor BiestableConcepto:** Dos genes que se reprimen mutuamente de forma cruzada. Si el gen 1 está encendido, apaga al gen 2. Si un estímulo externo enciende el gen 2, este apaga al gen 1 de forma permanente.Uso en ingeniería: Dota a las células de memoria. La célula puede recordar si estuvo expuesta a una toxina en el pasado, cambiando de color permanentemente.

**Interpretación de Ingeniería del GráficoPérdida de la Oscilación (Homeostasis Estable):**

 Si bajas la cooperatividad a n = 1, verás que las ondas desaparecen por completo. Las tres proteínas encuentran un punto de equilibrio plano. Esto demuestra matemáticamente por qué los ingenieros genéticos necesitan usar promotores moleculares fuertemente cooperativos en el laboratorio; de lo contrario, el reloj celular simplemente no funciona.Cambio de Frecuencia y Amplitud: Al alterar la Fuerza del Gen (\(\alpha \)) o la constante de afinidad (Kd), puedes sintonizar el circuito. 
 
 Esto equivale a cambiar los engranajes de un reloj mecánico para hacer que las proteínas oscilen más rápido o acumulen mayor concentración antes de decaer.

 **Interpretación de Ingeniería del Gráfico Pérdida de la Oscilación (Homeostasis Estable):**
 
  Si bajas la **cooperatividad a n = 1**, verás que las ondas desaparecen por completo. Las tres proteínas encuentran un punto de equilibrio plano. Esto demuestra matemáticamente por qué los ingenieros genéticos necesitan usar promotores moleculares fuertemente cooperativos en el laboratorio; de lo contrario, el reloj celular simplemente no funciona. 
  
  **Cambio de Frecuencia y Amplitud:** Al alterar la **Fuerza del Gen (\(\alpha \))** o la **constante de afinidad (Kd)**, puedes sintonizar el circuito. Esto equivale a cambiar los engranajes de un reloj mecánico para hacer que las proteínas oscilen más rápido o acumulen mayor concentración antes de decaer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from ipywidgets import interact, FloatSlider, IntSlider

# Ecuaciones de un Represilador de 3 genes utilizando cinética de Hill
def represilador_model(y, t, alpha, alpha0, n, kd):
    # px, py, pz son las concentraciones de las 3 proteínas represoras
    px, py, pz = y
    
    # Ecuaciones diferenciales interconectadas de manera cíclica
    # La proteína Z reprime la producción de X, X reprime a Y, e Y reprime a Z
    dpx_dt = alpha / (1 + (pz / kd)**n) + alpha0 - px
    dpy_dt = alpha / (1 + (px / kd)**n) + alpha0 - py
    dpz_dt = alpha / (1 + (py / kd)**n) + alpha0 - pz
    
    return [dpx_dt, dpy_dt, dpz_dt]

def simular_represilador(alpha, n, kd):
    t = np.linspace(0, 50, 1000)
    
    # Condiciones iniciales ligeramente desequilibradas para romper la simetría inicial
    y0 = [10.0, 2.0, 1.0]
    alpha0 = 0.01  # Transcripción basal residual (goteo del gen apagado)
    
    sol = odeint(represilador_model, y0, t, args=(alpha, alpha0, n, kd))
    
    plt.figure(figsize=(10, 5))
    plt.plot(t, sol[:, 0], lw=2, color='blue', label='Proteína X')
    plt.plot(t, sol[:, 1], lw=2, color='orange', label='Proteína Y')
    plt.plot(t, sol[:, 2], lw=2, color='green', label='Proteína Z')
    
    plt.title('Reloj Biológico Sintético (El Represilador)', fontsize=14, fontweight='bold')
    plt.xlabel('Tiempo celular (UA)', fontsize=12)
    plt.ylabel('Concentración de Proteínas', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.legend(loc='upper right')
    plt.show()

# Controles interactivos de bioingeniería
interact(simular_represilador,
         alpha=FloatSlider(value=50.0, min=10.0, max=100.0, step=5.0, description='Fuerza Gen (α):'),
         n=IntSlider(value=3, min=1, max=5, step=1, description='Cooperat. (n):'),
         kd=FloatSlider(value=2.0, min=0.5, max=5.0, step=0.5, description='Const. Afin (Kd):'))


interactive(children=(FloatSlider(value=50.0, description='Fuerza Gen (α):', min=10.0, step=5.0), IntSlider(va…

<function __main__.simular_represilador(alpha, n, kd)>